# Module 2.0 — Why Multi-Agent? Motivation & Architecture Patterns

> **First module of the MAS (Multi-Agent Systems) Part.**
> Day 1 built a single agent that can reflect on its own mistakes. The MAS Part builds *teams* of agents — each a specialist, each holding a scoped set of tools, each evaluated against a scoped output. This module is the bridge: why we need the shape change, which architectures are worth knowing, and a 30-line CrewAI hello-world to prove the plumbing talks to your local Ollama box.

```
  Day 1 :  one LLM  ->  ReAct loop  ->  5 tools  ->  Reflection / Reflexion
                                                             |
                                                             v
  MAS   :  >=2 LLMs ->  role-based  ->  scoped tools  ->  team output
```

Every Day 1 building block is reused. What changes is the **shape**: instead of one agent being everything, we split the work across specialists who hand off to each other.


## 1. The wall we just hit

At the end of Module 1.5 we ran the full Physics Research Assistant on a verbatim task:

> *"Search our paper corpus for recommended parameters, then run a quick simulation at three temperatures (below, at, and above T_c) on a 32×32 lattice. Report magnetization and energy at each temperature, and check whether results are consistent with theoretical predictions."*

Three clean attempts. No crashes. Plumbing solved. And yet `Accepted on attempt: None` — all three attempts compressed the three-part task into *one simulation at one temperature* and returned a bare float to `final_answer(...)`. The Critic caught it each time. Reflexion distilled three correct lessons into memory. None of that saved the run.

The root cause is not a bug in the tool layer and not a bug in the reflection layer. It is that a single qwen2.5:7b, given five tools and a three-subgoal prompt, cannot reliably hold all three subgoals in working memory for long enough to execute them in one ReAct chain. **Reflection is feedback *after* failure — it cannot prevent the failure from happening in the first place.**

The structural fix is to stop asking one agent to be everything. Give each subgoal its own agent, its own system prompt, and its own tools. That is what the MAS Part is about.


## 2. Four architecture patterns worth knowing

Before picking one, it helps to see the menu.

### 2.1 Sequential / Pipeline

```
  [Task] --> [Agent A] --output--> [Agent B] --output--> [Agent C] --> [Result]
```

Each agent transforms the previous output. Linear, predictable, easy to debug. **Physics analogue:** experimental data-processing chain — trigger → reconstruction → analysis. Each stage has one job and a well-defined input/output.

### 2.2 Hierarchical / Orchestrator

```
                  [Manager agent]
                /       |       \
          [Spec A]  [Spec B]  [Spec C]
```

A manager decomposes a task and delegates. Good when subtask structure is dynamic (the manager decides who to call based on the question). **Physics analogue:** a PI allocating work to postdocs and grad students based on the question of the day. Harder to run on small models — the manager has to reason about which specialist to call, and qwen2.5:7b-class models delegate unreliably.

### 2.3 Debate / Adversarial

```
  [Proposer] <---> [Critic]    ==>    [Judge] --> [Verdict]
```

Two agents argue from opposing positions; a third resolves. Excellent for catching systematic errors neither side would spot alone. **Physics analogue:** theorist vs experimentalist confrontation — *"your data disagrees with my model"* forces a deeper look. We already built a degenerate version of this in Module 1.4 (Actor vs Critic); the difference is that in a real debate *both* sides have tools and can bring new evidence.

### 2.4 Swarm / Peer-to-peer

```
  [Agent] <---> [Agent] <---> [Agent]
     ^                           ^
     |___________________________|
```

No designated roles; agents self-organize via broadcast. **Physics analogue:** emergent order in many-body systems — no one spin is in charge, but the ensemble finds a state. Powerful in principle, much harder to debug; we will not use it in this course.

---

For the MAS Part we use **Sequential** (this module), **Sequential-with-shared-tools** (Module 2.1), and **Debate** (Module 2.2). Hierarchical and Swarm are on the reading list, not the build list.


## 3. Why CrewAI (and why not the others)

There are at least four actively-maintained multi-agent frameworks for Python: **CrewAI**, AutoGen, AG2 (the fork that kept going after AutoGen's pivot), and LangGraph. We pick CrewAI for three reasons:

1. **Role-based vocabulary matches physics.** A CrewAI `Agent` has a `role`, a `goal`, and a `backstory` — literally the three questions you ask when introducing someone in a research group. This framing saves a lot of explanation.
2. **Ollama-native.** CrewAI's `LLM` class takes `"ollama/qwen2.5:7b"` as a model string out of the box. No adapter layer, no LiteLLM configuration tricks beyond what we already did in Day 1.
3. **Actively maintained and stable.** AutoGen 0.x is in maintenance. AG2 is young. LangGraph is powerful but graph-first, which buys complexity we do not need for a three-agent physics team.

CrewAI is **not** smolagents and **not** MCP. Day 1's smolagents `CodeAgent` runs Python code in a sandbox; CrewAI's `Agent` calls an LLM and — in later modules — calls tools, but has no Python-code-execution ReAct loop. They are different abstractions. This is deliberate: the **building blocks** (LLM, prompt, tools, memory) are the same; the **composition** changes.


## 4. The three primitives: Agent, Task, Crew

CrewAI has exactly three user-facing abstractions. Once you have these internalized, you can read any CrewAI codebase.

| Primitive | What it is | Minimum fields |
|---|---|---|
| **Agent** | A configured LLM with a persona and (optionally) tools. | `role`, `goal`, `backstory`, `llm` |
| **Task**  | A unit of work assigned to one Agent, with a description and an expected_output contract. | `description`, `expected_output`, `agent` |
| **Crew**  | An ordered list of Tasks plus a process type (`sequential` or `hierarchical`). Calling `.kickoff()` runs it. | `agents`, `tasks`, `process` |

Two things carry straight over from Day 1:

- **`expected_output` IS the `final_answer` contract.** The string you put in `expected_output` is what the downstream agent (or a CrewAI-internal reviewer) reads. Vague `expected_output` → vague answer. Make it specific, the same way we made `final_answer` self-contained in Modules 1.4–1.5.
- **LLM is configured once, reused everywhere.** Same pattern as the `LiteLLMModel(...)` we defined from Module 1.3 onwards.

The next four code cells build one of each primitive in isolation, then compose them into a running Crew.


In [ ]:
# Silence CrewAI's OpenTelemetry exporter BEFORE importing crewai.
# CrewAI ships anonymous usage telemetry by default; on a local/offline
# box it retries and spams stderr with "Connection reset by peer" noise.
# Turning it off makes cell output readable and does not affect functionality.
import os
os.environ["OTEL_SDK_DISABLED"]        = "true"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

import warnings
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew, Process, LLM

# ---------------------------------------------------------------------------
# LLM configured once, reused for every Agent in this notebook.
# Same qwen2.5:7b / Ollama / temperature=0 convention as Day 1.
# ---------------------------------------------------------------------------
llm = LLM(
    model="ollama/qwen2.5:7b",
    base_url="http://localhost:11434",
    temperature=0.0,
)
print(f"LLM configured: {llm.model}")


In [ ]:
# ---------------------------------------------------------------------------
# Primitive #1 -- Agent.
# A persona + an LLM. No tools yet (tools come in Module 2.1).
# `allow_delegation=False`: this agent cannot hand work off to another.
# Small Ollama models are unreliable at delegation decisions, and a
# sequential pipeline does not need it.
# ---------------------------------------------------------------------------
theorist = Agent(
    role="Theorist",
    goal="State exact theoretical values for 2D Ising-model observables.",
    backstory=(
        "A statistical mechanics theorist. Terse. Quotes numbers to four "
        "decimals. Knows Onsager (1944) by heart."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=False,
)
print(f"Agent built: role={theorist.role!r}")
print(f"             goal={theorist.goal!r}")


In [ ]:
# ---------------------------------------------------------------------------
# Primitive #2 -- Task.
# description + expected_output contract + assigned agent.
# `expected_output` is the MAS-Part equivalent of our Day 1 final_answer
# contract: be specific about shape, because whoever reads this (downstream
# agent or user) will compare the actual output against it. Vague contract
# -> vague answer, exactly as we saw with the Module 1.4 Critic.
# ---------------------------------------------------------------------------
task_state_tc = Task(
    description=(
        "State Onsager's exact critical temperature for the 2D Ising model "
        "in dimensionless units (J=1, k_B=1) to four decimal places."
    ),
    expected_output=(
        "One sentence of the form: 'T_c = X.XXXX.' "
        "No commentary, no surrounding prose."
    ),
    agent=theorist,
)
print(f"Task built: {task_state_tc.description[:60]}...")


In [ ]:
# ---------------------------------------------------------------------------
# Primitive #3 -- Crew.
# Process.sequential runs tasks in list order, passing each task's output
# as context to the next task. With one task and one agent this is just
# "run the task" -- the minimal demonstration.
# ---------------------------------------------------------------------------
hello_crew = Crew(
    agents=[theorist],
    tasks=[task_state_tc],
    process=Process.sequential,
    verbose=False,
)

import time
t0 = time.time()
result = hello_crew.kickoff()
elapsed = time.time() - t0

print(f"Crew finished in {elapsed:.1f}s\n")
print("Result:")
print(str(result))


## 5. Hello-world: two physicists, sequential pipeline

Now a second agent. The simplest non-trivial team is two specialists in sequence, each producing a one-sentence output that the next one reads.

```
  [Task 1] --> [Theorist: state T_c]
                      |
                      v  (Theorist's output becomes context)
  [Task 2] --> [Experimentalist: propose a sim to test it]
                      |
                      v
               [Crew result]
```

No tools. No simulations. Each agent is one LLM call with a scoped role. This is the CrewAI equivalent of *Hello, World* — enough to prove the hand-off works, nothing more.

Watch the output closely: the Experimentalist's response should **reference the Theorist's T_c value**. That cross-reference is the hand-off — proof that Crew passed Task 1's output as context for Task 2. That is what a pipeline buys you over two isolated LLM calls.


In [ ]:
# ---------------------------------------------------------------------------
# Two-agent sequential pipeline.
# Theorist states T_c; Experimentalist proposes a simulation to test it.
# The Experimentalist should reference the Theorist's value in its answer --
# that cross-reference is the evidence that CrewAI passed the output along.
# ---------------------------------------------------------------------------
experimentalist = Agent(
    role="Experimentalist",
    goal="Propose concrete Monte Carlo parameters to test a theoretical claim.",
    backstory=(
        "A computational physicist. Picks specific lattice sizes, step counts, "
        "and algorithms. Never hand-waves. One sentence per recommendation."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

task_propose_sim = Task(
    description=(
        "Given the Theorist's T_c value in the preceding task context, "
        "propose the concrete parameters for a single Monte Carlo simulation "
        "that would confirm it numerically on the 2D Ising model. Specify "
        "a CONCRETE lattice size L (a number, e.g. 32 or 64), a CONCRETE "
        "number of MC measurement steps N (a number, e.g. 10000), and the "
        "algorithm (Metropolis or Wolff). One sentence. Do NOT leave L or N "
        "as placeholder letters -- pick numbers."
    ),
    expected_output=(
        "One sentence of the form: "
        "'Run a <NUMBER>x<NUMBER> lattice for <NUMBER> steps with <ALGORITHM> "
        "at T = X.XXXX.' where every <NUMBER> is an actual integer. "
        "Must cite the T value from the preceding task."
    ),
    agent=experimentalist,
)

two_agent_crew = Crew(
    agents=[theorist, experimentalist],
    tasks=[task_state_tc, task_propose_sim],
    process=Process.sequential,
    verbose=False,
)

t0 = time.time()
two_result = two_agent_crew.kickoff()
elapsed = time.time() - t0
print(f"Crew finished in {elapsed:.1f}s\n")

# tasks_output gives us per-task output, each with .agent (role) and .raw
print("--- Per-task outputs ---")
for i, out in enumerate(two_result.tasks_output, start=1):
    print(f"[Task {i} | {out.agent}]")
    print(f"  {out.raw.strip()}\n")

print("--- Final crew result (= last task's output) ---")
print(str(two_result).strip())


## 6. What we saw — and what 2.1 will add

What this hello-world proved:

- **CrewAI talks to Ollama.** No tricks, no extra adapters beyond Day 1.
- **`Process.sequential` hands Task 1's output to Task 2 as context.** The Experimentalist's answer references the Theorist's T_c — that reference is the evidence.
- **Two agents, two system prompts, two scoped outputs.** Neither agent was asked to do the other's job.

What it did **not** prove (deliberately):

- **Tool use.** Neither agent has any tools yet. The Theorist states T_c from memory; the Experimentalist proposes parameters from its backstory. **Module 2.1** plugs in our five Day-1 tools and shows a team *actually* running simulations and searching papers.
- **Disagreement handling.** A sequential pipeline cannot disagree with itself. **Module 2.2** builds the Debate pattern on top of 2.1's toolchain.
- **Cross-system communication.** Everything here runs in one Python process. **Module 2.3** (A2A protocol) wraps an agent as a network service so two different codebases can collaborate.

One honest caveat carried over from Day 1: CrewAI does not magically fix qwen2.5:7b's tendency to editorialize. You will sometimes see outputs that are longer than `expected_output` asked for. When that happens, the lever is almost always **tighten `expected_output`**, not change the LLM. Same lesson as Day 1's `final_answer` contract.


## 7. Exercises

Two cheap-leverage exercises. Both are small edits to the hello-world cell.

### Exercise 2.0.1 — Swap the Experimentalist's model

Change the Experimentalist's `llm=` to a stronger model (assuming you've pulled it — `ollama pull llama3.1:8b`):

```python
experimentalist = Agent(
    ...,
    llm=LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.0),
    ...
)
```

Re-run. Compare:
- Does the 8B Experimentalist reference the Theorist's T_c more reliably?
- Does the output stay closer to the one-sentence contract?
- How much longer does the second task take?

**Takeaway:** multi-agent lets you mix model sizes where each specialist needs them, without paying the stronger-model cost on every agent.

### Exercise 2.0.2 — Add a third agent (Data Analyst)

Append a third `Agent` and `Task` to the Crew:

```python
analyst = Agent(
    role="Data Analyst",
    goal="Predict the expected |m| from the Experimentalist's proposed simulation using Onsager's exact formula.",
    backstory=(
        "A computational physicist who knows |m|(T) = (1 - sinh(2/T)^(-4))^(1/8) "
        "for T < T_c, and m = 0 otherwise. Reports a single number."
    ),
    llm=llm,
    allow_delegation=False,
)
task_predict = Task(
    description=(
        "Given the Experimentalist's proposed temperature in the preceding context, "
        "compute the expected |m| using Onsager's formula (or state m = 0 if T >= T_c)."
    ),
    expected_output="One number to four decimals. Nothing else.",
    agent=analyst,
)
# then: agents=[theorist, experimentalist, analyst],
#       tasks=[task_state_tc, task_propose_sim, task_predict]
```

Does the Analyst correctly evaluate `(1 - sinh(2/T)^(-4))^(1/8)` in its head? (Spoiler: qwen2.5:7b will not be reliable at this. Which is exactly why **Module 2.1 will give it a calculator**.)
